# Comparison of Lomg-Term Savings Strategies

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Introduction

This notebook compares different savings strategies over time.
It calculates the total savings for each strategy and provides a comparison.

Initial startegies for comparison include:
 - LISA Stocks & Bonds
 - LISA Cash
 - ISA Stocks & Bonds
 - ISA Cash
Which utilise these savings accounts only.

Further mixed strategies can be added to compare combinations of these accounts.
- LISA & ISA Overspill (strategy that utilises LISA accounts to maximum bonus and spills over to ISA accounts)
- Mixed (combination of LISA and ISA accounts using dictionaries to define the allocation ratios)

Tuning on any manual decisions is also performed for optimal strategy selection.

## Parameters

Default control variables and allocation ratios for savings strategies

They include:
 - *STARTING_AMOUNT* : Initial amount of money in the savings account
 - *YEARLY_CONTRIBUTION* : Amount contributed to the savings account each year
 - *TIME_HORIZON* : Number of years the savings strategy is applied
 - *STOCKS_BONDS_INTEREST* : Annual interest rate for stocks & bonds accounts
 - *CASH_INTEREST*: Annual interest rate for cash accounts

In [5]:
DEFAULT_STARTING_AMOUNT = 0
DEFAULT_YEARLY_CONTRIBUTION = 6000
DEFAULT_TIME_HORIZON = 40
DEFAULT_STOCKS_BONDS_INTEREST = 0.075
DEFAULT_CASH_INTEREST = 0.042

STARTING_AMOUNT = DEFAULT_STARTING_AMOUNT
YEARLY_CONTRIBUTION = DEFAULT_YEARLY_CONTRIBUTION
TIME_HORIZON = DEFAULT_TIME_HORIZON
STOCKS_BONDS_INTEREST = DEFAULT_STOCKS_BONDS_INTEREST
CASH_INTEREST = DEFAULT_CASH_INTEREST

CONTROL_VARIABLES = {
    "STARTING_AMOUNT": STARTING_AMOUNT,
    "YEARLY_CONTRIBUTION": YEARLY_CONTRIBUTION,
    "TIME_HORIZON": TIME_HORIZON,
    "STOCKS_BONDS_INTEREST": STOCKS_BONDS_INTEREST,
    "CASH_INTEREST": CASH_INTEREST
}

# Default allocation ratios for mixed strategies
default_allocation = {
    "lisa_stocks_bonds": 0.25,
    "lisa_cash": 0.25,
    "isa_stocks_bonds": 0.25,
    "isa_cash": 0.25
}

stocks_bonds_allocation = {
    "lisa_stocks_bonds": 0.5,
    "lisa_cash": 0.0,
    "isa_stocks_bonds": 0.5,
    "isa_cash": 0.0
}

cash_allocation = {
    "lisa_stocks_bonds": 0.0,
    "lisa_cash": 0.5,
    "isa_stocks_bonds": 0.0,
    "isa_cash": 0.5
}

Below is a function for easily updating these values.

In [6]:
def display_control_variables():

    print("Current control variables:")
    for key, value in CONTROL_VARIABLES.items():
        print(f"{key}: {value}")
    print()

def update_control_variables(
    starting_amount=None,
    yearly_contribution=None,
    time_horizon=None,
    stocks_bonds_interest=None,
    cash_interest=None):

    if starting_amount is not None:
        CONTROL_VARIABLES["STARTING_AMOUNT"] = starting_amount
        STARTING_AMOUNT = starting_amount
    if yearly_contribution is not None:
        CONTROL_VARIABLES["YEARLY_CONTRIBUTION"] = yearly_contribution
        YEARLY_CONTRIBUTION = yearly_contribution
    if time_horizon is not None:
        CONTROL_VARIABLES["TIME_HORIZON"] = time_horizon
        TIME_HORIZON = time_horizon
    if stocks_bonds_interest is not None:
        CONTROL_VARIABLES["STOCKS_BONDS_INTEREST"] = stocks_bonds_interest
        STOCKS_BONDS_INTEREST = stocks_bonds_interest
    if cash_interest is not None:
        CONTROL_VARIABLES["CASH_INTEREST"] = cash_interest
        CASH_INTEREST = cash_interest
    display_control_variables()

def reset_control_variables():
    
    global STARTING_AMOUNT, YEARLY_CONTRIBUTION, TIME_HORIZON, STOCKS_BONDS_INTEREST, CASH_INTEREST
    STARTING_AMOUNT = DEFAULT_STARTING_AMOUNT
    YEARLY_CONTRIBUTION = DEFAULT_YEARLY_CONTRIBUTION
    TIME_HORIZON = DEFAULT_TIME_HORIZON
    STOCKS_BONDS_INTEREST = DEFAULT_STOCKS_BONDS_INTEREST
    CASH_INTEREST = DEFAULT_CASH_INTEREST

    CONTROL_VARIABLES["STARTING_AMOUNT"] = DEFAULT_STARTING_AMOUNT
    CONTROL_VARIABLES["YEARLY_CONTRIBUTION"] = DEFAULT_YEARLY_CONTRIBUTION
    CONTROL_VARIABLES["TIME_HORIZON"] = DEFAULT_TIME_HORIZON
    CONTROL_VARIABLES["STOCKS_BONDS_INTEREST"] = DEFAULT_STOCKS_BONDS_INTEREST
    CONTROL_VARIABLES["CASH_INTEREST"] = DEFAULT_CASH_INTEREST
    display_control_variables()


## Strategies

In [ ]:
def cash(
    time: int = TIME_HORIZON,
    starting_amount: float = 0,
    yearly_contribution: float = YEARLY_CONTRIBUTION,
    ):
    
    portfolio = starting_amount
    for _ in range(time):
        portfolio += yearly_contribution
    return round(portfolio, 2)

def lisa_stocks_bonds(
    interest: float = 0.075,
    time: int = TIME_HORIZON,
    starting_amount: float = 0,
    yearly_contribution: float = YEARLY_CONTRIBUTION,
    ):
    
    portfolio = starting_amount
    for _ in range(time):
        portfolio += yearly_contribution
        portfolio += 0.25 * yearly_contribution if yearly_contribution <= 4000 * 0.25 else 4000 * 0.25
        portfolio *= (1 + interest)
    return round(portfolio, 2)

def lisa_cash(
        interest: float = 0.042,
        time: int = TIME_HORIZON,
        starting_amount: float = 0,
        yearly_contribution: float = YEARLY_CONTRIBUTION,
        ):
    
    return round(lisa_stocks_bonds(
        interest=interest,
        time=time,
        starting_amount=starting_amount,
        yearly_contribution=yearly_contribution
    ), 2)
    
def isa_stocks_bonds(
        interest: float = 0.075,
        time: int = TIME_HORIZON,
        starting_amount: float = 0,
        yearly_contribution: float = YEARLY_CONTRIBUTION,
        ):
    
    portfolio = starting_amount
    for _ in range(time):
        portfolio += yearly_contribution
        portfolio *= (1 + interest)
    return round(portfolio, 2)

def isa_cash(
        interest: float = 0.042,
        time: int = TIME_HORIZON,
        starting_amount: float = 0,
        yearly_contribution: float = YEARLY_CONTRIBUTION,
        ):
    
    return round(isa_stocks_bonds(
        interest=interest,
        time=time,
        starting_amount=starting_amount,
        yearly_contribution=yearly_contribution
    ), 2)

def lisa_isa_overspill(
        time: int = TIME_HORIZON,
        starting_amount: float = 0,
        yearly_contribution: float = YEARLY_CONTRIBUTION,
        interest: float = 0.042,
        ):

    holdings = {
        "lisa": starting_amount if starting_amount <= 4000 else 4000,
        "isa": starting_amount - 4000 if starting_amount > 4000 else 0
    }

    for _ in range(time):
        holdings["lisa"] += yearly_contribution if holdings["lisa"] + yearly_contribution <= 4000 else 4000 - holdings["lisa"]
        overspill = yearly_contribution - (4000 - holdings["lisa"]) if holdings["lisa"] < 4000 else yearly_contribution
        holdings["isa"] += overspill

        holdings["lisa"] *= (1 + interest)
        holdings["isa"] *= (1 + interest)



    return round(holdings["lisa"] + holdings["isa"], 2)

def mixed(
        allocation: dict = default_allocation,
        time: int = TIME_HORIZON,
        starting_amount: float = 0,
        yearly_contribution: float = YEARLY_CONTRIBUTION
        ):


    portfolio = {
        key: starting_amount * allocation[key] for key in allocation
    }

    # Dictionary mapping strategy names to their corresponding functions
    func_dict = {
        "cash": cash,
        "lisa_stocks_bonds": lisa_stocks_bonds,
        "lisa_cash": lisa_cash,
        "isa_stocks_bonds": isa_stocks_bonds,
        "isa_cash": isa_cash,
        "lisa_isa_overspill": lisa_isa_overspill
    }

    for _ in range(time):
        for key in portfolio:
            portfolio[key] += func_dict[key](
                time=1,
                starting_amount=portfolio[key],
                yearly_contribution=yearly_contribution * allocation[key]
            ) - portfolio[key]


    return round(sum(portfolio.values()), 2)

In [ ]:
def run_simulation():
    results = {}
    for strategy in ["cash", "lisa_stocks_bonds", "lisa_cash", "isa_stocks_bonds", "isa_cash", "lisa_isa_overspill"]:
        results[strategy] = mixed(
            allocation={strategy: 1},
            time=TIME_HORIZON,
            starting_amount=0,
            yearly_contribution=YEARLY_CONTRIBUTION
        )
    return results